NAME: Liya Elizabeth Raju
REGISTER NO: 25MML1009

In [1]:
!pip install pyspark

This cell installs the `pyspark` library, which is essential for working with Apache Spark in Python environments like Colab.

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("FileStreamingWordCount") \
    .getOrCreate()

spark

This cell imports `SparkSession` and initializes a new SparkSession named 'FileStreamingWordCount'. The `SparkSession` is the entry point to programming Spark with the Dataset and DataFrame API.

In [3]:
import os

input_dir = "stream_input"

# Create directory if it doesn't exist
os.makedirs(input_dir, exist_ok=True)

print("Input directory created:", input_dir)

Input directory created: stream_input


This cell imports the `os` module and creates a directory named `stream_input`. This directory will be used to simulate a data stream by adding new files to it, which Spark will then process.

In [4]:
# Add new files to simulate streaming
with open("stream_input/file1.txt", "w") as f:
    f.write("hello world hello spark")

with open("stream_input/file2.txt", "w") as f:
    f.write("spark streaming is powerful")

This cell simulates incoming data by creating two text files (`file1.txt` and `file2.txt`) inside the `stream_input` directory. These files contain sample sentences that Spark will later read as a stream.

In [10]:
from pyspark.sql.functions import explode, split

# Read streaming data from the folder
lines = spark.readStream \
    .format("text") \
    .load(input_dir)

This cell sets up a Spark streaming DataFrame (`lines`) to read data from the `stream_input` directory. It uses the `text` format, meaning each line in the files will be treated as a record in the stream.

In [11]:
# Split lines into words
words = lines.select(
    explode(
        split(lines.value, " ")
    ).alias("word")
)

# Count words
word_counts = words.groupBy("word").count()

This cell processes the streaming `lines` DataFrame. It uses Spark SQL functions `split` to break each line into words and `explode` to create a new row for each word. Finally, it groups the words and counts their occurrences.

In [12]:
query = word_counts.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("word_counts_table")

This cell configures the output of the streaming query. It specifies that the word counts should be outputted in 'complete' mode (all counts updated each trigger), formatted as an in-memory table named 'word_counts_table'.

In [8]:
query.start()

This cell starts the configured streaming query. Spark will now continuously monitor the `stream_input` directory for new files and update the `word_counts_table` accordingly.

In [13]:
import time
time.sleep(5)  # wait for Spark to process

spark.sql("SELECT * FROM word_counts_table").show()

+---------+-----+
|     word|count|
+---------+-----+
| powerful|    1|
|    hello|    2|
|streaming|    1|
|       is|    1|
|    spark|    2|
|    world|    1|
+---------+-----+



This cell waits for 5 seconds to allow Spark to process the initial files in the stream. Then, it queries the `word_counts_table` (the in-memory table where streaming results are stored) using SQL and displays the current word counts.